# Two numbers together

**Two numbers together -- scatter plots, and what to do when they pile up.**

> **The question this chart answers:** "does x have anything to do with y?"

**What it shows:**

- scatter first, statistics second -- always look before you fit
- overplotting, and three fixes: transparency, smaller marks, binning
- a trend line helps, but only if you say what kind it is
- the same correlation can come from very different pictures

---

*Chapter:* `choosing` — which chart answers which question  
*Run the cells in order.* Every figure is also written to `viz/output/choosing/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save, students

# Where save() files this lesson's output: viz/output/choosing/
LESSON = "choosing/relationship"


## The data

`students()` has study hours, an exam score, and a group label — two quantities plus a category, the standard scatter plot shape.


In [ ]:
data = students()


## 1. The basic scatter, with and without a trend

Scatter first, statistics second. The right-hand panel adds a least-squares line **and says so in the legend**: an unlabelled line could be a regression, a LOESS smooth, or someone's opinion.


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

left.scatter(data["hours"], data["score"], s=18, color="#4C72B0")
left.set(xlabel="hours studied", ylabel="exam score", title="Just the points")

right.scatter(data["hours"], data["score"], s=18, color="#4C72B0", alpha=0.6)
slope, intercept = np.polyfit(data["hours"], data["score"], 1)
xs = np.linspace(data["hours"].min(), data["hours"].max(), 100)
right.plot(xs, slope * xs + intercept, color="#C44E52", lw=2,
           label=f"least squares: {slope:.1f} points per hour")
right.legend()
right.set(xlabel="hours studied", ylabel="exam score",
          title="With a line -- and the line is labelled")

fig.tight_layout()
save(fig, LESSON, "scatter");


## 2. Overplotting: when there are too many points

40,000 points is more marks than pixels, so the default panel is saturated — it shows the outline of the data and nothing about its density. Each of the three fixes trades something different away.


In [ ]:
rng = np.random.default_rng(11)
n = 40_000
x = rng.normal(0, 1, n)
y = x * 0.6 + rng.normal(0, 0.8, n)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))

axes[0].scatter(x, y, s=20)
axes[0].set_title("Default: a solid blob")

axes[1].scatter(x, y, s=20, alpha=0.02)
axes[1].set_title("alpha=0.02: density appears")

axes[2].scatter(x, y, s=0.5, alpha=0.3)
axes[2].set_title("Tiny marks")

hb = axes[3].hexbin(x, y, gridsize=45, cmap="Blues")
axes[3].set_title("hexbin: count per cell")
fig.colorbar(hb, ax=axes[3], label="count")

for ax in axes:
    ax.set_xlabel("x")
axes[0].set_ylabel("y")

fig.suptitle(f"{n:,} points. The first panel is not a chart, it is an ink stain.",
             fontsize=11)
fig.tight_layout()
save(fig, LESSON, "overplotting");


## 3. The same correlation, four different truths

Anscombe's quartet is the argument for this whole chapter in one figure: four datasets with the same mean, variance, correlation and regression line, and four completely different stories.


In [ ]:
# Anscombe's quartet: four datasets with the same mean, variance, correlation
# and regression line. This is why you plot before you summarise.
quartet = {
    "I":   ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
            [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
            [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
            [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
            [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharex=True, sharey=True)
for ax, (name, (xs_, ys_)) in zip(axes, quartet.items()):
    xs_, ys_ = np.array(xs_), np.array(ys_)
    ax.scatter(xs_, ys_, color="#4C72B0", s=35)
    m, b = np.polyfit(xs_, ys_, 1)
    line = np.array([3, 20])
    ax.plot(line, m * line + b, color="#C44E52", lw=1.5)
    ax.set_title(f"{name}:  r = {np.corrcoef(xs_, ys_)[0, 1]:.2f}")
    ax.set_xlim(2, 20)

fig.suptitle("Anscombe's quartet: identical means, variances, correlation and "
             "regression line", fontsize=11)
fig.tight_layout()
save(fig, LESSON, "anscombe");


## Rules of thumb

```text
Look at the picture before you trust the number.
  few points     -> plain scatter
  many points    -> alpha, smaller marks, or hexbin
  adding a line  -> say which line it is
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. In section 1, fit a quadratic (`np.polyfit(..., 2)`) instead of a line. Does it fit better, and is 'better fit' a good enough reason to use it here?
2. In section 2, try `alpha=0.2` and `alpha=0.005`. Alpha tuning depends on n — what happens to the right alpha if you drop n to 2,000?
3. Compute `np.mean`, `np.var` and `np.corrcoef` for all four Anscombe sets and print them. How many digits do they agree to?


In [ ]:
# your turn


---

**Previous:** [`choosing/distribution`](distribution.ipynb)  
**Next:** [`choosing/change_over_time`](change_over_time.ipynb)
